In [8]:
# Install dependencies (run if needed)
# !pip install tensorflow opencv-python matplotlib numpy pillow mediapipe

In [9]:
import os
from pathlib import Path
import json
import os
from pathlib import Path
import json

DATA_DIR = r"C:\Users\sasin\OneDrive\Desktop\Strabismus\archive (2)\STRABISMUS"
EXTS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

def count_images(root: str = DATA_DIR):
    root_p = Path(root)
    if not root_p.exists():
        print(f"Dataset root not found: {root}")
        return {}
    counts = {}
    for d in sorted([p for p in root_p.iterdir() if p.is_dir()]):
        imgs = [p for p in d.iterdir() if p.suffix in EXTS]
        counts[d.name] = len(imgs)
    return counts

counts = count_images()
print('Counts per class:', counts)

manifest_path = Path(DATA_DIR).parent / 'strabismus_manifest_summary.json'
with open(manifest_path, 'w') as f:
    json.dump(counts, f)
print('Wrote summary to', manifest_path)


Counts per class: {'ESOTROPIA': 100, 'EXOTROPIA': 104, 'HYPERTROPIA': 102, 'HYPOTROPIA': 93, 'NORMAL': 110}
Wrote summary to C:\Users\sasin\OneDrive\Desktop\Strabismus\archive (2)\strabismus_manifest_summary.json


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7,1.3],
    shear_range=0.1)
train_data = datagen.flow_from_directory(
    DATA_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', subset='training')
val_data = datagen.flow_from_directory(
    DATA_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', subset='validation')

print('Detected classes:', train_data.class_indices)


Found 409 images belonging to 5 classes.
Found 100 images belonging to 5 classes.
Detected classes: {'ESOTROPIA': 0, 'EXOTROPIA': 1, 'HYPERTROPIA': 2, 'HYPOTROPIA': 3, 'NORMAL': 4}


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

base_model = tf.keras.applications.MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model.trainable = True
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(train_data.class_indices), activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25
)
model.summary()

c:\Users\sasin\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 48s 2s/step - accuracy: 0.1956 - loss: 2.0941 - val_accuracy: 0.1500 - val_loss: 2.1487
Epoch 2/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.2249 - loss: 1.9389 - val_accuracy: 0.1800 - val_loss: 2.0706
Epoch 3/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.2249 - loss: 1.9169 - val_accuracy: 0.1600 - val_loss: 1.9417
Epoch 4/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 28s 2s/step - accuracy: 0.2396 - loss: 1.8749 - val_accuracy: 0.2100 - val_loss: 1.9121
Epoch 5/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.2543 - loss: 1.7881 - val_accuracy: 0.2200 - val_loss: 1.8476
Epoch 6/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.2861 - loss: 1.7201 - val_accuracy: 0.2100 - val_loss: 1.7686
Epoch 7/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.2445 - loss: 1.7298 - val_accuracy: 0.1600 - val_loss: 1.8356
Epoch 8/25
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - accuracy: 0.2592 - loss: 1.7437 - val_accuracy: 0.2600 - val_loss:

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,199,569 (27.46 MB)

 Trainable params: 2,388,485 (9.11 MB)

 Non-trainable params: 34,112 (133.25 KB)

 Optimizer params: 4,776,972 (18.22 MB)

In [12]:
model.evaluate(val_data)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 387ms/step - accuracy: 0.3500 - loss: 1.6441


[1.644140362739563, 0.3499999940395355]

In [13]:
from pathlib import Path
import json

out_dir = Path('models')
out_dir.mkdir(parents=True, exist_ok=True)

model.save("models/strabismus_model.keras")
with open(out_dir / 'class_indices.json', 'w') as f:
    json.dump(train_data.class_indices, f)
print('Saved model and class indices to', out_dir)

Saved model and class indices to models


In [14]:
import numpy as np
from PIL import Image
import tensorflow as tf
import json
from pathlib import Path

class_indices_path = Path('models') / 'class_indices.json'
model_path = Path('models') / 'strabismus_model.keras'

if class_indices_path.exists():
    with open(class_indices_path, 'r') as f:
        class_indices = json.load(f)
    classes = [None] * len(class_indices)
    for k, v in class_indices.items():
        classes[v] = k
else:
    classes = None

def load_image(path, target_size=(224,224)):
    img = Image.open(path).convert('RGB')
    img = img.resize(target_size)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def predict_image(img_path):
    if not model_path.exists():
        print('No model found at', model_path)
        return
    model = tf.keras.models.load_model(str(model_path))
    x = load_image(img_path)
    pred = model.predict(x)[0]
    idx = int(np.argmax(pred))
    predicted_class = classes[idx] if classes is not None else str(idx)
    confidence = float(pred[idx]) * 100.0
    if predicted_class and predicted_class.lower() == 'normal':
        severity = 'none'
    else:
        if confidence < 60:
            severity = 'mild'
        elif confidence < 85:
            severity = 'moderate'
        else:
            severity = 'severe'
    print(f'Prediction: {predicted_class}')
    print(f'Confidence: {confidence:.2f}%')
    print(f'Severity (proxy): {severity}')